In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.decomposition import PCA
from umap import UMAP
import jscatter

In [4]:
%cd ..

f:\robusto\vqa_analysis


In [5]:
embedings_cache_path = "./embed_cache.pkl"
#load from pickle
import pickle
with open(embedings_cache_path, "rb") as f:
    embeddings_cache = pickle.load(f)

#loaded embeddings cache is a dict of text to embedding
print(f"Loaded embeddings cache with {len(embeddings_cache)} entries")

Loaded embeddings cache with 43047 entries


In [6]:
#load answers text cache
answers_text_cache_path = "./data/r2.csv"
df_answers = pd.read_csv(answers_text_cache_path, keep_default_na=False)
df_answers = df_answers[df_answers["REPETITION"] == 1]  # Filter out empty answers
print(f"Loaded answers text cache with {len(df_answers)} entries")
df_answers.head()

Loaded answers text cache with 10000 entries


,AGENT,VIDEO,BLOCK,QUESTION_NUM,REPETITION,ANSWER
0,human_lima_1,Robusto2_153,1,1,1,The ego vehicle is accelerating slowly because...
1,human_lima_2,Robusto2_153,1,1,1,The ego vehicle is turning to the right
2,human_lima_3,Robusto2_153,1,1,1,the ego vehicle brakes and steers slightly to ...
3,human_lima_4,Robusto2_153,1,1,1,Braking to yield
4,human_lima_5,Robusto2_153,1,1,1,The ego vehicle is moving forward while mainta...


In [7]:
#first reduce all embedding cache using pca to 2 dimensions and store then into a df with agent video question answer and embedding
embeddings = np.stack(df_answers["ANSWER"].map(embeddings_cache))
print(f"Embed matrix shape: {embeddings.shape}")

Embed matrix shape: (10000, 384)


In [8]:
coords_PCA = np.zeros((embeddings.shape[0], 2))
coords_UMAP = np.zeros((embeddings.shape[0], 2))
for block in [1, 2, 3, 4]:
    mask = df_answers["BLOCK"] == block
    print(f"Processing block {block} with {mask.sum()} entries")
    if mask.any():
        pca_block = PCA(n_components=2)
        #umap_block = UMAP(n_components=2,metric="cosine",local_connectivity = 1, n_jobs=16, random_state=42)
        coords_PCA[mask.values] = pca_block.fit_transform(embeddings[mask.values])
        #coords_UMAP[mask.values] = umap_block.fit_transform(embeddings[mask.values])


Processing block 1 with 2500 entries
Processing block 2 with 2500 entries
Processing block 3 with 2500 entries
Processing block 4 with 2500 entries


In [9]:
interactive = df_answers[["AGENT", "VIDEO","BLOCK", "QUESTION_NUM", "ANSWER"]].assign(pca_X=coords_PCA[:, 0], pca_Y=coords_PCA[:, 1])
interactive = interactive.assign(umap_X=coords_UMAP[:, 0], umap_Y=coords_UMAP[:, 1])

# --- Merge vectorizado ---
def get_group_color(agent):
    if "human" in agent:
        if "nyc" in agent:
            return "nyc"
        else:
            return "lima"
    else:
        return "vlm"
    
color_map = {
    "nyc": "#00E5FF", # Cian neón (sustituye al azul oscuro que no se ve)
    "lima": "#FF3D00", # Naranja-Rojo vibrante
    "vlm": "#00FF00", # Verde eléctrico (el que ya tienes, funciona bien)
}

interactive["group"] = interactive["AGENT"].map(get_group_color)
interactive.head()

,AGENT,VIDEO,BLOCK,QUESTION_NUM,ANSWER,pca_X,pca_Y,umap_X,umap_Y,group
0,human_lima_1,Robusto2_153,1,1,The ego vehicle is accelerating slowly because...,0.418390,0.064284,0.0,0.0,lima
1,human_lima_2,Robusto2_153,1,1,The ego vehicle is turning to the right,0.522257,0.002749,0.0,0.0,lima
2,human_lima_3,Robusto2_153,1,1,the ego vehicle brakes and steers slightly to ...,0.454121,-0.045594,0.0,0.0,lima
3,human_lima_4,Robusto2_153,1,1,Braking to yield,0.047026,-0.055907,0.0,0.0,lima
4,human_lima_5,Robusto2_153,1,1,The ego vehicle is moving forward while mainta...,0.507037,-0.105778,0.0,0.0,lima


## Embeds reduced by NEW pipeline - PCA reduced

In [10]:
import pandas as pd
import numpy as np

BLOCK_TO_PLOT = int(input("Enter block number to plot (1-4): "))
df_block = interactive[interactive["BLOCK"] == BLOCK_TO_PLOT]

# Sample data
# Create an interactive scatter plot
scatter = jscatter.Scatter(
    data= df_block,  # Filter for block 1
    x='pca_X',
    y='pca_Y',
    color_by='group',
    color_map=color_map,
    opacity=0.85,
    background_color="#111111",
    height=640,   
    axes = False,
    title=f"PCA Scatter Plot for Block {BLOCK_TO_PLOT}",
    tooltip=True,
    tooltip_preview= "ANSWER",
    tooltip_preview_type="text",
    tooltip_properties=["pca_X","pca_Y","AGENT", "VIDEO", "QUESTION_NUM"],
    tooltip_histograms_size="large",
    tooltip_size="medium"
)
scatter.axes(labels=['PCA 1', 'PCA 2'])

print(f"Displaying scatter plot  of Block {BLOCK_TO_PLOT}...")
scatter.show()

Displaying scatter plot  of Block 1...
